In [1]:
# load packages
import os
import regionmask
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
# data path
snow_path = "/home/mgurazada/snowdroughts_thirst/data/era5_monthly/era5_snow_vars.nc"

t2m_path = "/home/mgurazada/snowdroughts_thirst/data/era5_monthly/era5_monthly_t2m_19502024.nc"
ppt_path = "/home/mgurazada/snowdroughts_thirst/data/era5_monthly/era5_monthly_precipitation_1950_2024.nc"
eto_path = "/home/mgurazada/snowdroughts_thirst/data/era5_monthly/era5_monthly_eto_1950_2024.nc"

In [3]:
snow_vars = xr.open_dataset(snow_path)

# Calculate swe 
snow_vars['sd'].attrs
swe = (snow_vars['sd'] * 1000).rename("swe") # m w.e. → kg m-2 (≡ mm w.e.)
swe.attrs.update({
    "units": "kg m-2",
    "long_name": "Snow Water Equivalent (from snow depth in m w.e.)"
})

In [4]:
# open the data
t2m = xr.open_dataset(t2m_path)
t2m

<xarray.Dataset> Size: 4GB
Dimensions:     (valid_time: 900, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 7kB 1950-01-01 ... 2024-12-01
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    number      int64 8B ...
    expver      (valid_time) <U4 14kB ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 4GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-06T21:07 GRIB to CDM+CF via cfgrib-0.9.1...

In [5]:
# rename coordinates for t2m
swe = swe.rename({
    "valid_time": "time",
    "latitude": "lat",
    "longitude": "lon"
})

# rename coordinates for t2m
t2m = t2m.rename({
    "valid_time": "time",
    "latitude": "lat",
    "longitude": "lon"
})

In [6]:
# these data have variable "days" instead of "valid_time" and require different decoding
ppt = xr.open_dataset(ppt_path, decode_times=False)
eto = xr.open_dataset(eto_path, decode_times=False)

In [7]:
# create monthly timestamps (900 months starting Jan 1950)
times = pd.date_range("1950-01-01", periods=ppt.sizes['time'], freq="MS")

# assign to both datasets
ppt = ppt.assign_coords(time=("time", times))
eto = eto.assign_coords(time=("time", times))

print(ppt.time[0].values, ppt.time[-1].values)
print(eto.time[0].values, eto.time[-1].values)

1950-01-01T00:00:00.000000000 2024-12-01T00:00:00.000000000
1950-01-01T00:00:00.000000000 2024-12-01T00:00:00.000000000


In [8]:
# shift lon from 0–360 → −180–180 and sort
swe['lon'] = ((swe['lon'] + 180) % 360) - 180
t2m['lon'] = ((t2m['lon'] + 180) % 360) - 180
ppt['lon'] = ((ppt['lon'] + 180) % 360) - 180
eto['lon'] = ((eto['lon'] + 180) % 360) - 180

In [9]:
swe = swe.sortby('lon')
t2m = t2m.sortby('lon')
ppt = ppt.sortby('lon')
eto = eto.sortby('lon')

In [10]:
# flips lat +90 → -90 to −90 → +90
if float(swe.lat[0]) > float(swe.lat[-1]):
    swe = swe.sortby('lat')
    t2m = t2m.sortby('lat')
    ppt = ppt.sortby(
        'lat')
    eto = eto.sortby('lat')

In [11]:
# convert SWE (Dataarray) to Dataset and standardize to match t2m format
if isinstance(swe, xr.DataArray):
    swe = swe.to_dataset(name="swe")
    
swe

<xarray.Dataset> Size: 4GB
Dimensions:  (time: 900, lat: 721, lon: 1440)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1950-01-01 1950-02-01 ... 2024-12-01
  * lat      (lat) float64 6kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * lon      (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.2 179.5 179.8
    number   int64 8B ...
    expver   (time) <U4 14kB ...
Data variables:
    swe      (time, lat, lon) float32 4GB 1e+04 1e+04 1e+04 ... 0.0 0.0 0.0

In [13]:
# make all data have uniform coordinates and dimensions

def make_uniform(ds: xr.Dataset,
                 main_var_name: str,
                 rename_vars: dict | None = None,
                 drop_coords: tuple = ()):
    ds = ds.copy()

    # rename main variable 
    if rename_vars:
        keep = {k: v for k, v in rename_vars.items() if k in ds.data_vars}
        if keep:
            ds = ds.rename(keep)

    # clean up auxiliary coords
    to_drop = [c for c in drop_coords if c in ds.coords]
    if to_drop:
        ds = ds.drop_vars(to_drop, errors="ignore")

    # standardize coord dtypes (float64 for lat/lon)
    ds = ds.assign_coords(lat=ds.lat.astype("float64"),
                          lon=ds.lon.astype("float64"))
    ds[main_var_name] = ds[main_var_name].transpose('time', 'lat', 'lon')
    return ds[[main_var_name]]

# apply to each dataset
swe = make_uniform(
    swe,
    main_var_name="swe",
    rename_vars=None,
    drop_coords=("number", "expver")
)

t2m = make_uniform(
    t2m,
    main_var_name="t2m",
    rename_vars=None,
    drop_coords=("number", "expver")
)

# PPT
ppt = make_uniform(
    ppt,
    main_var_name="ppt",
    rename_vars={"pr": "ppt"},
    drop_coords=("crs",)
)

# ETO
eto = make_uniform(
    eto,
    main_var_name="eto",
    rename_vars=None,
    drop_coords=("crs",)
)

# align (exact grid/time) and merge
swe, t2m, ppt, eto = xr.align(swe, t2m, ppt, eto, join="exact")
ds_all = xr.merge([swe, t2m, ppt, eto], join="exact")

# save compressed NetCDF
enc = {v: {"zlib": True, "complevel": 4, "shuffle": True} for v in ds_all.data_vars}
out = "/home/mgurazada/snowdroughts_thirst/data/era5_monthly/era5_monthly_processed/era5_data.nc"
os.makedirs(os.path.dirname(out), exist_ok=True)
ds_all.to_netcdf(out, encoding=enc)

print("Wrote:", out)
print(ds_all)

Wrote: /home/mgurazada/snowdroughts_thirst/data/era5_monthly/era5_monthly_processed/era5_data.nc
<xarray.Dataset> Size: 15GB
Dimensions:  (time: 900, lat: 721, lon: 1440)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1950-01-01 1950-02-01 ... 2024-12-01
  * lat      (lat) float64 6kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * lon      (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.2 179.5 179.8
Data variables:
    swe      (time, lat, lon) float32 4GB 1e+04 1e+04 1e+04 ... 0.0 0.0 0.0
    t2m      (time, lat, lon) float32 4GB ...
    ppt      (time, lat, lon) float32 4GB ...
    eto      (time, lat, lon) float32 4GB ...


In [ ]:
ds_all['ppt']

In [ ]:
ds_all['ppt'].isel(time=-1).plot(
    cmap="Blues",
    figsize=(14,5)
)

plt.title("Precipitation — Last Time Step")
plt.show()

In [ ]:
# natural earth land polygons
land = regionmask.defined_regions.natural_earth_v5_0_0.land_110

# create mask
mask = land.mask(ds_all)

# apply mask (keep land, drop ocean)
ppt_land = ds_all['ppt'].isel(time=-1).where(~mask.isnull())

In [ ]:
plt.figure(figsize=(14,5))

ppt_land.plot(
    cmap="Blues",
    robust=True,
    vmax=300 
)

plt.title("Precipitation — Land Only")
plt.show()

In [ ]:
swe['swe']

In [ ]:
# kg m-2 == mm water equivalent
swe = swe["swe"]

# plot latest month
swe.isel(time=-1).plot(
    cmap="Blues",
    figsize=(14,5),
    vmin=0,
    vmax=400   # adjust
)
plt.title("Snow Water Equivalent (mm) — latest timestep")
plt.show()

In [ ]:
swe.to_netcdf("/home/mgurazada/snowdroughts_thirst/data/era5_monthly/era5_monthly_processed/swe.nc")